In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('proact_preprocessed_S1.csv')
df.head()

,subject_id,ALSFRS_Delta,Q1_Speech,Q2_Salivation,Q3_Swallowing,Q4_Handwriting,Q5_Cutting,Q6_Dressing_and_Hygiene,Q7_Turning_in_Bed,Q8_Walking,Q9_Climbing_Stairs,R_1_Dyspnea,R_2_Orthopnea,R_3_Respiratory_Insufficiency,Age_Base,Sex_Female,Treatment_Active,Bulbar_Onset,FVC_Base,BMI_Base
0,16497,0.0,4.0,4.0,4.0,3.0,3.0,2.0,1.0,1.0,0.0,4.0,4.0,4.0,0.016919,1.0,1.0,0.0,0.126976,-0.356485
1,53215,8.0,3.0,2.0,4.0,4.0,4.0,3.0,3.0,3.0,3.0,4.0,4.0,4.0,-1.013892,0.0,1.0,1.0,0.289065,-0.354520
2,53215,71.0,3.0,2.0,4.0,3.0,4.0,3.0,3.0,3.0,1.0,4.0,4.0,4.0,-1.013892,0.0,1.0,1.0,0.289065,-0.354520
3,53215,140.0,3.0,3.0,4.0,3.0,4.0,2.0,3.0,3.0,3.0,3.0,4.0,4.0,-1.013892,0.0,1.0,1.0,0.289065,-0.354520
4,53215,204.0,3.0,4.0,3.0,3.0,3.0,3.0,3.0,2.0,2.0,4.0,4.0,4.0,-1.013892,0.0,1.0,1.0,0.289065,-0.354520


In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from girth import grm_mml
from scipy.special import expit # Sigmoid function

# Suppress warnings for clean output
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. HELPER FUNCTIONS FOR IRT MAPPING
# ==========================================

def get_expected_item_score(theta, discrimination, thresholds):
    """
    Maps a latent theta to an expected raw score (0-4) based on GRM parameters.
    P(score >= k) = expit(a * (theta - b_k))
    """
    # Calculate probabilities of exceeding each threshold
    p_ge_1 = expit(discrimination * (theta - thresholds[0]))
    p_ge_2 = expit(discrimination * (theta - thresholds[1]))
    p_ge_3 = expit(discrimination * (theta - thresholds[2]))
    p_ge_4 = expit(discrimination * (theta - thresholds[3]))
    
    # Calculate probability of exact categories
    p4 = p_ge_4
    p3 = p_ge_3 - p_ge_4
    p2 = p_ge_2 - p_ge_3
    p1 = p_ge_1 - p_ge_2
    p0 = 1.0 - p_ge_1
    
    # Expected value formula: Sum of (category_value * probability)
    expected_score = (0 * p0) + (1 * p1) + (2 * p2) + (3 * p3) + (4 * p4)
    return expected_score

# ==========================================
# 2. DEFINE SCENARIOS
# ==========================================
# Because IRT requires multiple items to estimate a latent trait, 
# individual item predictions are derived from the Domain or Total models.

domains = {
    'Bulbar': ['Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing'],
    'Fine_Motor': ['Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene'],
    'Gross_Motor': ['Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs'],
    'Respiratory': ['R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency']
}

all_items = [item for sublist in domains.values() for item in sublist]
covariates = 'ALSFRS_Delta + Sex_Female + Treatment_Active + Age_Base + Bulbar_Onset + FVC_Base + BMI_Base'

results_list = []
gkf = GroupKFold(n_splits=5)

print("Starting IRT + Latent LMM Evaluation Pipeline...\n")

# ==========================================
# 3. RUN EVALUATIONS
# ==========================================

# We will run one large model for the Total Score, and separate models for Domains
scenarios_to_run = {'Total Score': all_items, **domains}

for target_name, items in scenarios_to_run.items():
    print(f"Fitting IRT + LMM for {target_name} ({len(items)} items)...")
    
    # --- Step A: Fit the IRT Model (Graded Response Model) ---
    # girth requires items as rows, participants as columns, and values starting at 1
    # Cast the float data to integers before transposing
    irt_data = df[items].astype(int).values.T + 1
    
    # Fit model (Marginal Maximum Likelihood)
    irt_results = grm_mml(irt_data)
    
    # Extract item parameters
    discriminations = irt_results['Discrimination']
    difficulties = irt_results['Difficulty'] # 2D array [items, thresholds]
    
    # Append the estimated theta to our dataframe
    df['theta'] = irt_results['Ability']
    
    # --- Step B: Full LMM Fit for AIC/BIC on Latent Space ---
    formula = f"theta ~ {covariates}"
    try:
        full_model = smf.mixedlm(formula, data=df, groups=df["subject_id"]).fit(reml=False, disp=False)
        aic = full_model.aic
        bic = full_model.bic
    except:
        aic, bic = np.nan, np.nan
        
    # --- Step C: Cross-Validation for Out-of-Sample RMSE on RAW Space ---
    item_rmses = {item: [] for item in items}
    aggregate_rmses = []
    
    for train_idx, test_idx in gkf.split(df, groups=df['subject_id']):
        train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]
        
        try:
            # 1. Train LMM on latent theta
            cv_model = smf.mixedlm(formula, data=train_df, groups=train_df["subject_id"]).fit(disp=False)
            
            # 2. Predict latent theta for new patients
            pred_theta = cv_model.predict(exog=test_df)
            
            # 3. Map predicted theta back to expected raw scores for each item
            expected_aggregate_score = np.zeros(len(test_df))
            
            for i, item in enumerate(items):
                # Calculate expected score for this specific item
                exp_item_score = get_expected_item_score(pred_theta, discriminations[i], difficulties[i])
                
                # Calculate and store Item-Level RMSE
                rmse_item = np.sqrt(mean_squared_error(test_df[item], exp_item_score))
                item_rmses[item].append(rmse_item)
                
                # Add to aggregate sum
                expected_aggregate_score += exp_item_score
            
            # Calculate Aggregate (Domain/Total) RMSE
            actual_aggregate = test_df[items].sum(axis=1)
            rmse_agg = np.sqrt(mean_squared_error(actual_aggregate, expected_aggregate_score))
            aggregate_rmses.append(rmse_agg)
            
        except Exception as e:
            continue
            
    # --- Step D: Store Results ---
    # Store Aggregate Results
    results_list.append({
        'Level': 'Overall' if target_name == 'Total Score' else 'Domain',
        'Target': target_name,
        'Out-of-Sample RMSE': np.mean(aggregate_rmses),
        'Latent_AIC': aic,
        'Latent_BIC': bic
    })
    
    # Store Individual Item Results (only extracted from the Domain fits to avoid duplicates)
    if target_name != 'Total Score':
        for item in items:
            results_list.append({
                'Level': 'Individual Item',
                'Target': item,
                'Out-of-Sample RMSE': np.mean(item_rmses[item]),
                'Latent_AIC': np.nan, # AIC/BIC only apply to the aggregate latent model
                'Latent_BIC': np.nan
            })

# ==========================================
# 4. PRINT RESULTS
# ==========================================
evaluation_df = pd.DataFrame(results_list)

# Sort for clean display
evaluation_df['Level'] = pd.Categorical(evaluation_df['Level'], categories=['Individual Item', 'Domain', 'Overall'], ordered=True)
evaluation_df = evaluation_df.sort_values(['Level', 'Target'])

print("\n" + "="*85)
print("IRT + LATENT LMM: OUT-OF-SAMPLE PREDICTIVE ACCURACY ON RAW OBSERVED SCORES")
print("="*85)
format_dict = {'Out-of-Sample RMSE': '{:.3f}'.format, 'Latent_AIC': '{:.1f}'.format, 'Latent_BIC': '{:.1f}'.format}
print(evaluation_df.to_string(formatters=format_dict, index=False))

Starting IRT + Latent LMM Evaluation Pipeline...

Fitting IRT + LMM for Total Score (12 items)...
Fitting IRT + LMM for Bulbar (3 items)...
Fitting IRT + LMM for Fine_Motor (3 items)...
Fitting IRT + LMM for Gross_Motor (3 items)...
Fitting IRT + LMM for Respiratory (3 items)...

IRT + LATENT LMM: OUT-OF-SAMPLE PREDICTIVE ACCURACY ON RAW OBSERVED SCORES
          Level                        Target Out-of-Sample RMSE Latent_AIC Latent_BIC
Individual Item                     Q1_Speech              0.899        NaN        NaN
Individual Item                 Q2_Salivation              0.813        NaN        NaN
Individual Item                 Q3_Swallowing              0.795        NaN        NaN
Individual Item                Q4_Handwriting              1.076        NaN        NaN
Individual Item                    Q5_Cutting              1.208        NaN        NaN
Individual Item       Q6_Dressing_and_Hygiene              1.025        NaN        NaN
Individual Item             Q7_Turn

In [5]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error
from girth import grm_mml
from scipy.special import expit
from factor_analyzer import FactorAnalyzer
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. SETUP AND HELPER FUNCTIONS
# ==========================================
# IMPORTANT: Ensure your longitudinal dataframe is loaded as 'df'

domains = {
    'Bulbar': ['Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing'],
    'Fine_Motor': ['Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene'],
    'Gross_Motor': ['Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs'],
    'Respiratory': ['R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency']
}
all_items = [item for sublist in domains.values() for item in sublist]
scenarios = {'Total Score': all_items, **domains}

# Covariates (Ensure these match your actual dataframe columns)
covariates = 'ALSFRS_Delta + Sex_Female + Treatment_Active + Age_Base'
covariates_ar = 'Time_Since_Last_Visit + Sex_Female + Treatment_Active + Age_Base'

results_list = []

def get_expected_item_score(theta, discrimination, thresholds):
    """Maps latent theta back to expected 0-4 item score."""
    p_ge_1 = expit(discrimination * (theta - thresholds[0]))
    p_ge_2 = expit(discrimination * (theta - thresholds[1]))
    p_ge_3 = expit(discrimination * (theta - thresholds[2]))
    p_ge_4 = expit(discrimination * (theta - thresholds[3]))
    
    p4 = p_ge_4
    p3 = p_ge_3 - p_ge_4
    p2 = p_ge_2 - p_ge_3
    p1 = p_ge_1 - p_ge_2
    p0 = 1.0 - p_ge_1
    return (0 * p0) + (1 * p1) + (2 * p2) + (3 * p3) + (4 * p4)

# ==========================================
# BASELINE 1: NAIVE LMM (RAW SCORES)
# ==========================================
print("1. Fitting Naive LMM (In-Sample)...")

# Create aggregate targets
df['Total Score'] = df[all_items].sum(axis=1)
# for dom, items in domains.items():
#     df[dom] = df[items].sum(axis=1)

# targets_naive = {
#     'Overall': ['Total Score'], 
#     'Domain': list(domains.keys()), 
#     'Individual Item': all_items
# }

# for level, t_list in targets_naive.items():
#     for target in t_list:
#         formula = f"Q('{target}') ~ {covariates}" if ' ' in target else f"{target} ~ {covariates}"
#         try:
#             model = smf.mixedlm(formula, data=df, groups=df["subject_id"]).fit(reml=False, disp=False)
#             rmse = np.sqrt(mean_squared_error(df[target], model.fittedvalues))
#             results_list.append({'Model': '1. Naive LMM', 'Level': level, 'Target': target, 'In-Sample RMSE': rmse})
#         except Exception as e:
#             pass


# ==========================================
# BASELINE 2: IRT + LATENT LMM
# ==========================================
print("2. Fitting IRT + Latent LMM (In-Sample)...")

for target_name, items in scenarios.items():
    level = 'Overall' if target_name == 'Total Score' else 'Domain'
    
    # Fit IRT
    irt_data = df[items].astype(int).values.T + 1
    irt_results = grm_mml(irt_data)
    df['theta'] = irt_results['Ability']
    
    # Fit Latent LMM
    formula = f"theta ~ {covariates}"
    try:
        model = smf.mixedlm(formula, data=df, groups=df["subject_id"]).fit(reml=False, disp=False)
        fitted_theta = model.fittedvalues
        
        expected_agg = np.zeros(len(df))
        for i, item in enumerate(items):
            exp_item = get_expected_item_score(fitted_theta, irt_results['Discrimination'][i], irt_results['Difficulty'][i])
            expected_agg += exp_item
            
            # Save Individual Item RMSE only during the Domain loops to avoid duplicates
            if level == 'Domain':
                rmse_item = np.sqrt(mean_squared_error(df[item], exp_item))
                results_list.append({'Model': '2. IRT + LMM', 'Level': 'Individual Item', 'Target': item, 'In-Sample RMSE': rmse_item})
                
        actual_agg = df[items].sum(axis=1)
        rmse_agg = np.sqrt(mean_squared_error(actual_agg, expected_agg))
        results_list.append({'Model': '2. IRT + LMM', 'Level': level, 'Target': target_name, 'In-Sample RMSE': rmse_agg})
    except Exception as e:
        pass



# ==========================================
# 4. FORMAT FINAL COMPARISON TABLE
# ==========================================
eval_df = pd.DataFrame(results_list)

# Set logical order for the table
eval_df['Level'] = pd.Categorical(eval_df['Level'], categories=['Overall', 'Domain', 'Individual Item'], ordered=True)
eval_df = eval_df.sort_values(['Level', 'Target', 'Model'])

# Pivot the table so Models are columns, and Targets are rows
pivot_df = eval_df.pivot(index=['Level', 'Target'], columns='Model', values='In-Sample RMSE')

print("\n" + "="*85)
print("IN-SAMPLE RMSE ABLATION STUDY: BASELINE MODELS")
print("="*85)
print(pivot_df.to_string(float_format="{:.3f}".format))

1. Fitting Naive LMM (In-Sample)...
2. Fitting IRT + Latent LMM (In-Sample)...

IN-SAMPLE RMSE ABLATION STUDY: BASELINE MODELS
Model                                          2. IRT + LMM
Level           Target                                     
Overall         Total Score                           3.637
Domain          Bulbar                                0.904
                Fine_Motor                            1.042
                Gross_Motor                           1.111
                Respiratory                           1.214
Individual Item Q1_Speech                             0.490
                Q2_Salivation                         0.552
                Q3_Swallowing                         0.500
                Q4_Handwriting                        0.612
                Q5_Cutting                            0.508
                Q6_Dressing_and_Hygiene               0.685
                Q7_Turning_in_Bed                     0.786
                Q8_Walking       

In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error, mean_absolute_error, cohen_kappa_score
from girth import grm_mml
from scipy.special import expit
from factor_analyzer import FactorAnalyzer
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. SETUP & DATA PREP
# ==========================================
print("Loading data and setting up...")
# IMPORTANT: Load your actual patient data here
df = pd.read_csv('proact_preprocessed_S1.csv') 

domains = {
    'Bulbar': ['Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing'],
    'Fine_Motor': ['Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene'],
    'Gross_Motor': ['Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs'],
    'Respiratory': ['R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency']
}
all_items = [item for sublist in domains.values() for item in sublist]
scenarios = {'Total Score': all_items, **domains}

# Add aggregated true scores to dataframe
df['Total Score'] = df[all_items].sum(axis=1)
for dom, items in domains.items():
    df[dom] = df[items].sum(axis=1)

covariates = 'ALSFRS_Delta + Sex_Female + Treatment_Active + Age_Base'
covariates_ar = 'Time_Since_Last_Visit + Sex_Female + Treatment_Active + Age_Base'

results_list = []

def record_metrics(model_name, level, target, actual, expected, is_item=False):
    """Helper function to calculate and store all advanced metrics."""
    rmse = np.sqrt(mean_squared_error(actual, expected))
    mae = mean_absolute_error(actual, expected)
    
    if is_item:
        pred_cat = np.clip(np.round(expected), 0, 4)
        exact = np.mean(pred_cat == actual) * 100
        kappa = cohen_kappa_score(actual, pred_cat, weights='linear')
    else:
        exact, kappa = np.nan, np.nan
        
    results_list.append({
        'Model': model_name, 'Level': level, 'Target': target,
        'RMSE': rmse, 'MAE': mae, 'Exact Match %': exact, 'Weighted Kappa': kappa
    })

def get_irt_expected(theta, discrimination, thresholds):
    """Maps latent theta back to 0-4 scale."""
    p_ge = [expit(discrimination * (theta - th)) for th in thresholds]
    p4 = p_ge[3]
    p3 = p_ge[2] - p_ge[3]
    p2 = p_ge[1] - p_ge[2]
    p1 = p_ge[0] - p_ge[1]
    p0 = 1.0 - p_ge[0]
    return (0 * p0) + (1 * p1) + (2 * p2) + (3 * p3) + (4 * p4)

# # ==========================================
# # 1. BASELINE 1: NAIVE LMM
# # ==========================================
# print("Fitting 1. Naive LMM...")
# targets_naive = {'Overall': ['Total Score'], 'Domain': list(domains.keys()), 'Individual Item': all_items}

# for level, t_list in targets_naive.items():
#     for target in t_list:
#         formula = f"Q('{target}') ~ {covariates}" if ' ' in target else f"{target} ~ {covariates}"
#         try:
#             model = smf.mixedlm(formula, data=df, groups=df["subject_id"]).fit(reml=False, disp=False)
#             expected = model.fittedvalues
#             # Enforce clinical boundaries for Naive LMM metrics
#             max_val = 48 if level == 'Overall' else (12 if level == 'Domain' else 4)
#             expected = np.clip(expected, 0, max_val)
#             record_metrics('1. Naive LMM', level, target, df[target], expected, is_item=(level=='Individual Item'))
#         except: pass

# ==========================================
# 2. BASELINE 2: IRT + LATENT LMM
# ==========================================
print("Fitting 2. IRT + Latent LMM...")
for target_name, items in scenarios.items():
    level = 'Overall' if target_name == 'Total Score' else 'Domain'
    irt_data = df[items].astype(int).values.T + 1
    irt_results = grm_mml(irt_data)
    df['theta'] = irt_results['Ability']
    
    try:
        model = smf.mixedlm(f"theta ~ {covariates}", data=df, groups=df["subject_id"]).fit(reml=False, disp=False)
        expected_agg = np.zeros(len(df))
        
        for i, item in enumerate(items):
            exp_item = get_irt_expected(model.fittedvalues, irt_results['Discrimination'][i], irt_results['Difficulty'][i])
            expected_agg += exp_item
            if level == 'Domain': # Capture items during domain loops
                record_metrics('2. IRT + LMM', 'Individual Item', item, df[item], exp_item, is_item=True)
                
        record_metrics('2. IRT + LMM', level, target_name, df[target_name], expected_agg, is_item=False)
    except: pass

# # ==========================================
# # 3. BASELINE 3: DISCRETE-TIME FA
# # ==========================================
# print("Fitting 3. Discrete-Time FA...")
# df = df.sort_values(by=['subject_id', 'ALSFRS_Delta'])
# df['Delta_Lag'] = df.groupby('subject_id')['ALSFRS_Delta'].shift(1)
# df['Time_Since_Last_Visit'] = df['ALSFRS_Delta'] - df['Delta_Lag']

# for target_name, items in scenarios.items():
#     level = 'Overall' if target_name == 'Total Score' else 'Domain'
#     fa = FactorAnalyzer(n_factors=1, rotation=None)
#     fa.fit(df[items])
#     df['Factor'] = fa.transform(df[items])[:, 0]
    
#     df['Factor_Lag'] = df.groupby('subject_id')['Factor'].shift(1)
#     df_dyn = df.dropna(subset=['Factor_Lag', 'Time_Since_Last_Visit']).copy()
#     formula = f"Factor ~ Factor_Lag + {covariates_ar}"
    
#     try: # LMM with OLS Fallback
#         try:
#             model = smf.mixedlm(formula, data=df_dyn, groups=df_dyn["subject_id"])
#             fitted_factor = model.fit(method='lbfgs', reml=False, disp=False).fittedvalues
#         except:
#             fitted_factor = smf.ols(formula, data=df_dyn).fit().fittedvalues
            
#         expected_agg = np.zeros(len(df_dyn))
#         for i, item in enumerate(items):
#             exp_item = np.clip((fitted_factor * fa.loadings_[:, 0][i]) + df_dyn[items].mean().values[i], 0, 4)
#             expected_agg += exp_item
#             if level == 'Domain':
#                 record_metrics('3. Discrete FA', 'Individual Item', item, df_dyn[item], exp_item, is_item=True)
                
#         record_metrics('3. Discrete FA', level, target_name, df_dyn[target_name], expected_agg, is_item=False)
#     except: pass

# ==========================================
# 4. FORMAT AND PRINT OUTPUT
# ==========================================
eval_df = pd.DataFrame(results_list)
eval_df['Level'] = pd.Categorical(eval_df['Level'], categories=['Overall', 'Domain', 'Individual Item'], ordered=True)

# Helper to print pivot tables
def print_metric_table(metric_name, format_str):
    pivot = eval_df.pivot(index=['Level', 'Target'], columns='Model', values=metric_name)
    print("\n" + "="*80)
    print(f"BASELINE COMPARISON: {metric_name}")
    print("="*80)
    print(pivot.to_string(float_format=lambda x: format_str.format(x) if pd.notnull(x) else "-"))

print_metric_table('RMSE', "{:.3f}")
print_metric_table('MAE', "{:.3f}")
print_metric_table('Exact Match %', "{:.1f}%")
print_metric_table('Weighted Kappa', "{:.3f}")

Loading data and setting up...
Fitting 2. IRT + Latent LMM...

BASELINE COMPARISON: RMSE
Model                                          2. IRT + LMM
Level           Target                                     
Overall         Total Score                           3.637
Domain          Bulbar                                0.904
                Fine_Motor                            1.042
                Gross_Motor                           1.111
                Respiratory                           1.214
Individual Item Q1_Speech                             0.490
                Q2_Salivation                         0.552
                Q3_Swallowing                         0.500
                Q4_Handwriting                        0.612
                Q5_Cutting                            0.508
                Q6_Dressing_and_Hygiene               0.685
                Q7_Turning_in_Bed                     0.786
                Q8_Walking                            0.444
           